In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [55]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import json
import os
import random
from torchvision import datasets, transforms
# from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [44]:
# load MNIST data
transform = transforms.ToTensor()
train_data  = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)


In [45]:
# firstly see how the data look like and size of dataset
img, label = train_data[16]
print(img.size)
# plt.imshow(img)
print(f"label is {label}, and image size is {img.size}\n")
print(f"train data size {len(train_data)}, test data size {len(test_data)}")

<built-in method size of Tensor object at 0x7c969542e8f0>
label is 2, and image size is <built-in method size of Tensor object at 0x7c969542e8f0>

train data size 60000, test data size 10000


In [47]:
# process data get a train with 20 points, valid with 100 points rest of train are pooling points
# train and valid need random and balanced
len_train = len(train_data)
index_lst = []
for i in range(10):
  data_index = [index for index in range(len_train) if train_data[index][1] == i ]
  index_lst.append(data_index)

In [49]:

train_index = []
valid_index = []
for i in range(10):
  temp_choose = random.sample(index_lst[i], k=12)
  train_index.extend(temp_choose[:2])
  valid_index.extend(temp_choose[2:])
  index_lst[i] = [x for x in index_lst[i] if x not in temp_choose]


In [53]:
path = '/content/drive/MyDrive/UDL_DATA/'
train_file_name = 'train_indices_extend.json'
pool_file_name = 'pooling_indices_new.json' # 建议加上轮次
train_file_path = os.path.join(path, train_file_name)
pool_file_path = os.path.join(path, pool_file_name)
# with open(train_file_path, 'w') as f:
#     json.dump(train_index, f)
with open(train_file_path, 'r') as f:
    train_index = json.load(f)

In [64]:
print(len(train_index))
# index_lst= [i  for j in  index_lst for i in j]
index_lst
index_lst= [x for x in index_lst if x not in train_index]


20


In [63]:
# add repeatition on pooling layer

# pooling_index = index_lst
# pooling_choose = random.sample(index_lst, k=4000)
# temp = pooling_choose.copy()
# pooling_choose.extend(temp)
# random.shuffle(pooling_choose)

# full_path = os.path.join(path, pool_file_name)
# with open(pool_file_path, 'w') as f:
#     json.dump(pooling_choose, f)
with open(pool_file_path, 'r') as f:
    pooling_index = json.load(f)
print(len(pooling_index))


8000


In [65]:
class NewDataset(Dataset):
  def __init__(self, index):
    imgs = []
    labels = []
    for i in index:
      img, label = train_data[i]
      imgs.append(img)
      labels.append(label)
    self.imgs = torch.stack(imgs)
    self.labels = torch.tensor(labels)

  def __len__(self):
    return len(self.imgs)

  def __getitem__(self, idx):
    return self.imgs[idx], self.labels[idx]



In [66]:
class BaseCNN(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.covn1 =nn.Conv2d(in_channels=1, out_channels=32,  kernel_size=4)
    self.covn2 = nn.Conv2d(in_channels=32, out_channels=32,  kernel_size=4)
    self.max_pool = nn.MaxPool2d(kernel_size=2)
    self.dropout_layer1 = nn.Dropout(p=0.25)
    self.flatten = nn.Flatten()
    self.dense_layer1 = nn.Linear(in_features=3872, out_features=128)
    self.dropout_layer2 = nn.Dropout(p=0.5)
    self.dense_layer2 = nn.Linear(in_features=128, out_features=10)

  def forward(self, x):
    x = F.relu(self.covn1(x))
    x = F.relu(self.covn2(x))
    x = self.max_pool(x)
    x = self.dropout_layer1(x)
    x = self.flatten(x)
    x = F.relu(self.dense_layer1(x))
    x = self.dropout_layer2(x)
    x = self.dense_layer2(x)
    # x = F.softmax(x, dim=1)
    return x

In [67]:
# write train function
def train_model(trainData):
    train_loader  = DataLoader(trainData, batch_size=128, shuffle=True)
    model = BaseCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
    for epoch in tqdm.tqdm(range(50)):
      total_loss = 0
      for i, (imgs, labels) in (enumerate(train_loader)):
        optimizer.zero_grad()
        imgs = imgs.to(device)
        labels = labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
      avg_loss = total_loss / len(train_loader)
      # if epoch % 79 == 0:
      #   print(f"/n epoch{epoch}, train_loss is {avg_loss:.4f}.")

    return model



In [89]:

# from PIL.Image import new
def expection_term(prob_matrix):
    # E(H(y1:n|..))
    # prob_matrix NxKxC
    temp1 = -torch.sum(prob_matrix * torch.log2(prob_matrix+ 1e-15), dim=-1) # NxK
    temp2 = torch.sum(temp1, dim=0) # K
    entropy = torch.mean(temp2) # scalar
    return entropy

def entropy_term(P_n_1, P_n):
    #H(y1... yn) P_n_1:   C^(N-1) xK, P_n: C xk
    _, k = P_n_1.shape
    matr = P_n_1 @ P_n.t() /k
    matr = -torch.sum(matr*torch.log2(matr + 1e-15))
    entropy = torch.sum(matr)
    return entropy

def all_pool_prob(model, poolingData, drop_out_iter=100):
    # batch size B, class: C dropout_iter K, Pooldata_len = P
    # collect prob information
    pooling_loader =DataLoader(poolingData, batch_size=200, shuffle=False)
    total_probs =[]
    for i ,(imgs, labels) in (enumerate(pooling_loader)):
        each_probs =[]
        for _ in range(drop_out_iter):
          with torch.no_grad():
              output =model(imgs.to(device))
              pred_prob = F.softmax(output, dim=1) # BxC
              each_probs.append(pred_prob)
        temp= torch.stack(each_probs) # K x B xC
        total_probs.append(temp.transpose(0, 1)) #BxKxC

    total_probs = torch.cat(total_probs, dim=0) # PxKxC
    return total_probs

def bald_scorce(total_probs):
    avg_prob = total_probs.mean(dim=1) # PxC
    bald_expect_term   = - torch.sum(avg_prob*torch.log2(avg_prob + 1e-15), dim=1) # P,
    temp = - torch.sum(total_probs*torch.log2(total_probs + 1e-15), dim=-1) # PxK
    bald_entropy_term = torch.mean(temp) # P,
    bald_score = bald_expect_term - bald_entropy_term
    return bald_score

def Batchbald_with_greedy(model, poolingData, pooling_index, batch_bald_size=3):
    curr_selected_index = []
    len_pool = len(pooling_index)
    # P_n_1 = total_probs[0, :].t() # Cxk
    C= 10
    K= 100
    P= len_pool
    total_probs= all_pool_prob(model, poolingData, drop_out_iter=100) # PxKxC
    assert total_probs.shape == (P, K, C)


    # select first base on BALD
    bald_score = bald_scorce(total_probs) # P
    first_index = torch.argmax(bald_score).item()
    curr_selected_index.append(first_index)
    P_n_1 = total_probs[first_index, :].t() #CxK
    assert P_n_1.shape == (C, K)
    j = 1
    while len(curr_selected_index) != batch_bald_size:
        curr_optimal = 0
        curr_index = 0
        for i in range(len_pool):
            if i not in curr_selected_index:
                P_n = total_probs[i, :].t() # CxK
                assert P_n.shape == (C, K)
                temp_lst = curr_selected_index.copy()
                temp_lst.append(i)
                temp_total_probs = total_probs[temp_lst, :]
                # expection_term need NxKxC, total_probs PxKxC
                expection_value = expection_term(temp_total_probs)
                entropy_term_value = entropy_term(P_n_1, P_n)
                batch_bald_score =entropy_term_value- expection_value
                if batch_bald_score > curr_optimal:
                    curr_optimal = batch_bald_score
                    curr_index = i
        new_prob = total_probs[curr_index, : ].t() # CxK
        assert new_prob.shape == (C, K)
        #  new p_n :  cxk pn-1 :c^n-1 x k
        # temp = torch.zero()
        curr_selected_index.append(curr_index)
        assert P_n_1.shape == (C**j, K)
        temp_matr = P_n_1.unsqueeze(1) * new_prob.unsqueeze(0) # C^n-1 x C x K
        # print(temp_matr.shape)
        P_n_1 = temp_matr.flatten(0, 1)
        # print(P_n_1.shape)
        j +=1
    new_pooling_index = [v for i, v in enumerate(pooling_index) if i not in set(curr_selected_index) ]
    return curr_selected_index, new_pooling_index



In [69]:
# def greedy_algo(acq_size=4, poolingData, pooling_index):
#     candidate = set{}
#     len_pool = len(pooling_index)
#     pooling_loader  = DataLoader(poolingData, batch_size=1, shuffle=False)
#     for i in range(acq_size):
#         curr_data = 0
#         cur_score = 0
#         for i in range(len_pool):
#             if i not in candidate:
#                 temp_set = candidate.copy()
#                 temp_set.add(i)
#                 batch_bald = (pooling_loader, temp_set, model)
#              if batch_bald > cur_score:
#                 curr_data = i
#                 cur_score = batch_bald
#         candidate.add(curr_data)
#     return candidate.tolist()





In [91]:
# calculate test accuracy
def test_accuracy( model, test_data, device=device):
  test_loader = DataLoader(test_data, batch_size=200, shuffle=False)
  model.eval()
  correct = 0
  n_data = len(test_data)
  with torch.no_grad():
    for imgs, labels in test_loader:
      imgs = imgs.to(device)
      labels = labels.to(device)
      outputs = model(imgs)
      predicted = outputs.argmax(dim=1)
      correct += (predicted == labels).sum().item()
  return correct / n_data

In [94]:
# one experiment
#return pooling_index new train_index
def run_once(train_index, pooling_index, test_data=test_data,  batch_bald_size=4):
  print(f"curr size of train_data {len(train_index)}, curr size of pooling data {len(pooling_index)}  ")
  trainData = NewDataset(index=train_index)
  # vaildData = NewDataset(index=valid_index)
  poolingData =  NewDataset(index=pooling_index)
  model = train_model(trainData=trainData)
  new_trainData_index, new_pool_index = Batchbald_with_greedy(model, poolingData, pooling_index, batch_bald_size=batch_bald_size)
  print(len(set(new_trainData_index)))
  train_index.extend(new_trainData_index)
  test_ac = test_accuracy(model, test_data)
  print(f"test accuracy is {test_ac}")
  return train_index, new_pool_index, test_ac


In [ ]:
# number of experiement
pooling_index_temp = pooling_index.copy()
train_index_temp = train_index.copy()

n_experiement = 150
test_accuracy_lst = []
for i in range(n_experiement):
    train_index_temp, pooling_index_temp, test_ac= run_once(train_index_temp, pooling_index_temp, batch_bald_size=4)
    test_accuracy_lst.append(test_ac)
    # if i ==2 :
    #   break



curr size of train_data 20, curr size of pooling data 8000  


100%|██████████| 50/50 [00:00<00:00, 359.40it/s]


4
test accuracy is 0.5225
curr size of train_data 24, curr size of pooling data 7996  


100%|██████████| 50/50 [00:00<00:00, 363.90it/s]


4
test accuracy is 0.5505
curr size of train_data 28, curr size of pooling data 7992  


...

100%|██████████| 50/50 [00:00<00:00, 210.60it/s]


4
test accuracy is 0.7685
curr size of train_data 76, curr size of pooling data 7944  


100%|██████████| 50/50 [00:00<00:00, 193.82it/s]


4
test accuracy is 0.7346
curr size of train_data 80, curr size of pooling data 7940  


100%|██████████| 50/50 [00:00<00:00, 188.12it/s]


In [78]:
def save_accuracy(file_name, accuracy_lst):
  path = '/content/drive/MyDrive/UDL_DATA/'
  path = path + file_name
  with open(path, 'w') as f:
      for item in accuracy_lst:
          f.write(f"{item}\n")



In [75]:
save_accuracy("batch_BALD_a_301iter_batchsize3.txt",test_accuracy_lst )